# PTIT Library: bóc tách PDF hàng loạt

Trước khi chạy: **Add Input** → dataset `kaggle-upload`; Session options → **Accelerator: GPU T4 x2**, **Internet: On**.

Chạy lần lượt từng ô. Bị ngắt giữa chừng thì chạy lại ô 1 → ô 2 → ô 4: file đã xong được bỏ qua.

In [ ]:
# Ô 1: tìm dataset, chép code ra thư mục ghi được
import glob, os, shutil
found = glob.glob("/kaggle/input/**/chunk_config.json", recursive=True)
assert found, "Chưa gắn dataset: bấm Add Input ở cột phải"
BUNDLE = os.path.dirname(found[0])
print("Dataset:", BUNDLE, "| số PDF:", len(glob.glob(BUNDLE + "/pdfs/*.pdf")))
shutil.copytree(BUNDLE + "/worker", "/kaggle/working/worker", dirs_exist_ok=True)
os.chdir("/kaggle/working/worker")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ - bật Accelerator GPU")

In [ ]:
# Ô 2: cài thư viện (1–3 phút)
!pip install -q -r /kaggle/working/worker/requirements-kaggle.txt

Ô 3 chạy thử 3 file: phải ra `OK` và **không** có câu "Chưa sinh được embedding".

In [ ]:
# Ô 3: chạy thử 3 file
!cd /kaggle/working/worker && python -m batch.kaggle_extract --bundle "$BUNDLE" --out /kaggle/working/output --device cuda --limit 3

In [ ]:
# Ô 4: chạy toàn bộ (có thể mất 1–2 giờ)
!cd /kaggle/working/worker && python -m batch.kaggle_extract --bundle "$BUNDLE" --out /kaggle/working/output --device cuda

In [ ]:
# Ô 5: tóm tắt kết quả
import pandas as pd
report = pd.read_csv("/kaggle/working/output/report.csv", encoding="utf-8-sig")
print(report["result"].value_counts())
report[report["result"] != "OK"][["file", "result", "note"]]

In [ ]:
# Ô 6: nén kết quả để tải về (cột phải → Output → output.zip)
!cd /kaggle/working && rm -f output.zip && zip -qr output.zip output && ls -lh output.zip
from IPython.display import FileLink
FileLink("/kaggle/working/output.zip")